# 0. Overview: from a query to a cited cohort

spatialdata-db is a catalog of public spatial omics datasets, each converted to a SpatialData store. The Python client
reads the catalog, selects datasets, loads them, and records what you used so that you and others can load the
same data again and cite where it came from.

The overview runs the whole path once, on three small Visium sections of human pancreatic tumours: find datasets,
build a cohort, load one dataset, join the tables into one `AnnData`, freeze a manifest and write the citations.
Each of the other tutorials covers one part of it in depth, with the mistakes we have seen people make.

## Install

The client is not on PyPI yet, so install it from GitHub. The `viz` extra adds the plotting packages that the
tutorials use.

```bash
pip install "spatialdata-db[viz] @ git+https://github.com/theislab/spatialdata-db-client@v0.1.2"
```

The tutorials download about 50 MB of data in total, into the cache described in {doc}`02_load_and_inspect`.
Tutorial 1 also has two cells that download the gene index (about 900 MB) and need a lot of memory.

## The recipe

| step | what it is for | tutorial |
|---|---|---|
| `sddb.Catalog(version=...)` | open one fixed release of the catalog | [3](03_cohorts_and_manifests.ipynb) |
| `cat.query`, `cat.search` | find datasets by facet, range, free text or gene | [1](01_find_and_filter.ipynb) |
| `cohort.where`, `cat.cohort`, `cat.collection` | build the set of datasets you want | [3](03_cohorts_and_manifests.ipynb) |
| `summary`, `groupby`, `licenses` | check the selection before you download | [1](01_find_and_filter.ipynb), [4](04_citations_and_licences.ipynb) |
| `ds.elements`, `ds.to_sdata` | look inside one dataset and load it | [2](02_load_and_inspect.ipynb) |
| `cohort.to_anndata` | one table across datasets, with matching gene ids | [3](03_cohorts_and_manifests.ipynb) |
| `cohort.freeze`, `verify` | record the exact datasets and check them later | [3](03_cohorts_and_manifests.ipynb) |
| `cohort.citations` | cite the studies behind the data | [4](04_citations_and_licences.ipynb) |

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import sddb

work = Path(tempfile.mkdtemp())  # the files this page writes go here

## A fixed catalog release

Every tutorial opens the catalog with `sddb.Catalog(version="2026-10-28")`. The catalog is published as a dated
file with every release, and the version picks that file. Without `version`, `Catalog()` reads `catalog-latest`,
which changes with each release: datasets get new versions and new uids, some are withdrawn and others are added.
Pinning the release makes every count and uid on the tutorial pages match what you get when you run them. In your own
work, pin the release for the same reason, and record it with your results. {doc}`03_cohorts_and_manifests`
counts how much changed between two releases.

In [ ]:
cat = sddb.Catalog(version="2026-10-28")
cat

## Find datasets

`query` filters on exact facet values. Here, Visium sections of human pancreas:

In [ ]:
pancreas = cat.query(organism="human", technology="Visium", tissue="pancreas")
print(pancreas)
pancreas.groupby("disease")

## Build a cohort

A query result is already a cohort, and you can narrow it further. Here we keep the stores under 10 MB, so the
page runs fast, and check the summary and the licenses before anything is downloaded.

In [ ]:
cohort = pancreas.where(pancreas.members["size_bytes"] < 10e6)
print(cohort.summary())
print(cohort.licenses())
cohort.to_df()[["uid", "disease", "n_obs", "size_bytes", "collections"]]

## Load one dataset

`elements()` lists what a store holds without downloading it, and `to_sdata()` downloads it into the cache once
and opens it as `SpatialData`.

In [ ]:
smallest = min(cohort, key=lambda ds: ds.size_bytes)
print(list(smallest.elements()))
sdata = smallest.to_sdata()
sdata

## Join the tables into one `AnnData`

`to_anndata` reads only the table of each member, not the images, and joins them. `gene_key="symbol"` puts every
member's genes under gene symbols first, so that sources which name genes differently share columns. `obs["uid"]` records which dataset each spot came from.

In [ ]:
adata = cohort.to_anndata(gene_key="symbol")
print(adata)
adata.obs["uid"].value_counts()

In [ ]:
counts = np.log1p(np.asarray(adata[:, "COL1A1"].X.todense()).ravel())
fig, axes = plt.subplots(1, len(cohort), figsize=(4 * len(cohort), 4))
for ax, uid in zip(axes, cohort.members["uid"], strict=True):
    rows = (adata.obs["uid"] == uid).to_numpy()
    xy = np.asarray(adata.obsm["spatial"], dtype=float)[rows]
    dots = ax.scatter(xy[:, 0], xy[:, 1], s=4, c=counts[rows], cmap="viridis", vmin=0, vmax=counts.max())
    ax.set_title(f"{uid}\n{rows.sum()} spots")
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_axis_off()
fig.colorbar(dots, ax=axes, label="COL1A1, log(1 + counts)", shrink=0.8)
plt.show()

`COL1A1` marks the fibrous stroma that surrounds pancreatic tumours. The three sections share one colour scale, so
you can compare them directly.

## Freeze a manifest

`freeze()` records the exact members, the catalog release and the sha256 of the catalog file. Write the manifest
next to your results. `from_manifest` rebuilds the same cohort later, and `verify()` checks that the catalog file
has not changed and every store still resolves.

In [ ]:
manifest = cohort.freeze()
path = manifest.write(work / "pancreas.json")
restored = sddb.SpatialDataCohort.from_manifest(sddb.CohortManifest.load(path))
restored.verify()
print(manifest.hash)
print("same members:", list(restored.members["uid"]) == list(cohort.members["uid"]))

## Write the citations

`citations()` writes one BibTeX entry per study. A study with an entry in the published bibliography gets that
entry, and any other study gets a fallback entry that cites its dataset page, with a warning that tells you to
check the fallback entries before you publish.

In [ ]:
bib = cohort.citations(work / "pancreas.bib")
print(bib.read_text())

## Where to go next

| tutorial | question |
|---|---|
| [1. Find and filter datasets](01_find_and_filter.ipynb) | which datasets match, and why does a search return more than I expected? |
| [2. Load and inspect a dataset](02_load_and_inspect.ipynb) | what is in a store, and how do I load only what I need? |
| [3. Cohorts and manifests](03_cohorts_and_manifests.ipynb) | how do I combine datasets, and get the same ones back next month? |
| [4. Citations and licenses](04_citations_and_licences.ipynb) | whom do I cite, and what may I do with the data? |

The [use cases](../use_cases/index.md) are longer analyses built on the same steps, and the
{doc}`../reference/api` lists every class and function.